# PCS956 - Time Series for ML - Anomalies, Concept Drift, and Multivariate/Nonlinear Dependence

This lecture extends the previous material by focusing on:
- abnormal behaviour (anomalies) in time series;
- changing behaviour over time (concept drift) and model maintenance;
- multivariate and nonlinear dependence, contextual features, and explainability in ML-style models.

Building on Lectures TS1 and TS2, we move from typical behaviour and forecasting performance to:
- detecting unusual events and regime changes;
- monitoring models over time in drifting environments;
- exploiting and interpreting interactions across variables.

By the end of the lecture, students should be able to:
- describe different types of anomalies and approaches to detection in time series;
- explain concept drift and basic strategies for monitoring and retraining models;
- recognise risks of spurious correlations and the need for transformations and decompositions;
- understand high-level tools for multivariate dependence and Granger-style predictive causality;
- appreciate explainability and research trends, with a sceptical attitude towards new methods;
- connect these ideas to the time-series mini-project, including simulation-based investigations.

A separate Companion C notebook provides self-contained code templates related to this lecture, including:
- residual-based anomaly flagging;
- block-wise performance monitoring for concept drift;
- cross-correlation and Granger-style multivariate tools;
- supervised learning with exogenous features and simple feature-importance based explainability;
- ML models on lagged + context features (for example Random Forest and gradient boosting);
- walk-forward evaluation patterns for ML models under concept drift.

TS3 focuses on concepts and examples; Companion C focuses on reusable code.


## 1. Recap and positioning

Lecture TS1 covered:
- foundations of time series, EDA, decomposition, stationarity;
- data quality, anomalies, and modelling pitfalls;
- an initial discussion of concept drift and the scientific mindset for temporal data.

Lecture TS2 focused on:
- forecasting baselines and classical models;
- supervised-learning formulations for forecasting;
- temporal validation, quarantine gaps, and leakage;
- evaluation on levels vs differences and residuals.

Lecture TS3 builds on these ideas to address:
- behaviour that deviates from typical patterns (anomalies);
- behaviour that changes over time (concept drift);
- complex multivariate and nonlinear relationships, contextual features, and explainability;
- implications for the time-series mini-project, including simulation-based studies.


### 1.1 From typical behaviour to anomalies and drift

Up to now, we have concentrated on typical or expected behaviour:
- identifying trend, seasonality, and dependence;
- building baselines and models for forecasting;
- evaluating models on levels and differences under temporal validation.

This lecture shifts the focus to:
- abnormal behaviour (anomalies) that may correspond to faults, attacks, or rare events;
- changing behaviour over time (concept drift) that can undermine previously good models;
- multivariate and nonlinear dependence, where interactions across variables are informative but
  can also be misleading.

These themes connect directly to cross-module questions about:
- causality: what changed and why, and what interventions might help;
- explainability: why a model raises an anomaly flag or forecasts a given pattern;
- uncertainty: how confident we can be in predictions under drift and rare events.


In [ ]:
# TODO: optional short recap example of forecasts and residuals from earlier lectures

## 2. Anomaly detection in time series

Anomaly detection aims to identify unusual behaviour relative to some notion of normal time-series
dynamics. It builds naturally on:
- typical patterns of trend, seasonality, and dependence (TS1 and TS2);
- residuals from forecasting models or decompositions;
- data-quality and rare-event considerations (TS1 Section 3).


### 2.1 Types of anomalies

Common anomaly types in time series include:

- point anomalies:
  - isolated spikes or drops;
  - single time points with values far from typical levels;
- contextual anomalies:
  - values that are unusual given the current context (for example time of day, season, regime);
  - a temperature that is normal in summer but anomalous in winter;
- collective anomalies:
  - unusual subsequences or patterns over time;
  - sustained deviations, abnormal oscillations, or atypical sequences of events.

Distinguishing these types matters for:
- the choice of detection method;
- interpretation and response, especially in operational settings.


### 2.2 Approaches to anomaly detection

Approaches can be broadly grouped into ways of formalising what counts
as 'normal' and then flagging observations that are implausible under
that notion of normal behaviour:

- threshold-based methods:
  - define simple rules (for example flag if value exceeds mean plus $k$ standard deviations);
  - easy to implement but may miss contextual or collective anomalies;
- model-based methods:
  - fit a forecasting or reconstruction model;
  - define anomalies as points or sequences with large residuals or reconstruction errors;
  - examples include ARIMA residual thresholds, forecast-error sequences from rolling one-step-ahead
    evaluations, or state-space model residuals;
- unsupervised and semi-supervised methods:
  - clustering of subsequences;
  - autoencoder-style models that learn typical patterns and flag large reconstruction errors;
  - density-estimation approaches that identify low-probability observations.

All of these rely on some definition of normal behaviour. Mis-specification of normality or changes
in normal behaviour over time (drift) can lead to false alarms or missed anomalies. In particular:

- residual-based methods only work as intended if the residuals under normal conditions are
  approximately stationary noise (for example roughly constant mean and variance, no strong
  autocorrelation);
- if the model is mis-specified (for example missing a level shift or new seasonality), large
  residuals may reflect model error rather than genuine anomalies.

Companion C Section 1 provides a simple template for residual-based anomaly flags:
- fit a baseline forecasting model on a period believed to be relatively normal;
- compute multi-step-ahead forecast residuals on a later evaluation window (forecasting from the end
  of the training period);
- flag observations whose residuals are unusually large relative to the residual variability in that
  calibration window.

This can be combined with simple, physically motivated checks on levels and jumps.


### 2.3 Data quality versus genuine anomalies

Anomalies must be interpreted in the context of data quality, as emphasised in TS1 Section 3:

- data-quality issues:
  - missing values, sensor faults, calibration changes;
  - communication failures, logging errors, unit mismatches;
- genuine anomalies:
  - rare events (for example extreme storms, market crashes, equipment failures);
  - regime shifts due to interventions, policy changes, or physical changes.

Differentiating these requires:
- domain knowledge (what readings are physically plausible);
- awareness of system changes (maintenance, upgrades, policy interventions);
- careful use of metadata and contextual information.

Limited training data and absent extremes restrict our ability to detect and interpret anomalies:
- models trained only on quiet periods may fail in extreme regimes;
- thresholds calibrated on typical behaviour may produce many false alarms when drift occurs.

These issues connect directly to concept drift (Section 3) and to multivariate and contextual structure
(Section 4), where additional variables and regime indicators can help interpret anomalies.


In [ ]:
# See Companion C Section 1 for a self-contained residual-based anomaly example on a toy series.

## 3. Concept drift and model maintenance

Concept drift concerns changes in the data-generating process or in
the relationships between inputs and outputs over time. For deployed
time-series models, drift is often the main reason why initially good
models degrade. TS1 Section 7.3 introduced this idea; here we develop
it further and connect it to monitoring and maintenance strategies.

### 3.1 What is concept drift?

Concept drift refers to situations where the statistical properties of
the data, or the relationship between predictors and targets for a
given prediction task, change over time. In probabilistic terms, the
joint distribution $P(X_t, Y_t)$ evolves over time, so that either the
marginal distribution of inputs $P(X_t)$, the conditional distribution
$P(Y_t \mid X_t)$, or both, are no longer stable. This can involve:

- changes in marginal distributions:
  - inputs $X_t$ or outputs $Y_t$ change in distribution over time;
  - for example different typical ranges, variances, or seasonal patterns in different periods;
- changes in conditional relationships:
  - the mapping from inputs to outputs, $P(Y_t \mid X_t)$, evolves;
  - models that were accurate become misaligned with the new relationship;
- sudden regime shifts:
  - interventions, policy changes, failures, or new operating modes;
  - abrupt changes in behaviour that break previous patterns.

These changes can be gradual or abrupt, and they may affect:
- forecast accuracy;
- anomaly detection thresholds;
- interpretation of multivariate dependence.


### 3.2 Stable versus drifting domains

It is helpful to distinguish between:

- relatively stable domains:
  - some physical systems and engineered processes where underlying laws and operating conditions
    change slowly;
  - models can sometimes be trained once and remain valid for long periods, with occasional
    recalibration;
- drift-heavy domains:
  - markets, social behaviour, climate-affected processes, or systems with frequent policy changes
    and interventions;
  - models require regular attention and may need frequent retraining or adaptation.

In drift-heavy domains, drift detection and adaptation are not optional extras but part of the core
modelling pipeline. The ARIMA drift example in TS1 Section 7.4 illustrated how a model that performs
acceptably in one regime can fail as soon as the mean or variance shifts.

These ideas reinforce the core message that time series models deployed in drifting environments must
be monitored and retrained as needed.


### 3.3 Monitoring models over time

Monitoring involves tracking:

- performance over time:
  - plots of error metrics (MAE, RMSE, optionally $R^2$ where appropriate) across successive periods;
  - detection of degradation relative to baselines;
- residual behaviour:
  - changes in residual distributions (mean, variance, skewness);
  - evolving autocorrelation structures indicating new dynamics;
- input statistics:
  - shifts in mean, variance, and correlations among inputs;
  - changes in the frequency of extreme values or regime indicators.

A simple practical approach is to evaluate forecasting performance on
successive time blocks using an expanding training window, and plot
metrics such as RMSE or MAE over time. Companion C Section 2 provides
a template for this kind of block-wise performance monitoring for
ARIMA-type models, and Section 7 extends the idea to walk-forward
evaluation for ML models on lagged + exogenous features.

Effective monitoring:
- combines quantitative metrics with domain knowledge;
- looks for both abrupt changes and gradual trends;
- integrates anomaly flags with drift signals, rather than treating them in isolation.


### 3.4 Adaptation and retraining strategies

Common adaptation strategies include:

- rolling-window retraining:
  - refit models on the most recent data within a moving window;
  - discard very old data that may no longer be relevant;
- periodic retraining schedules:
  - refit models at fixed intervals (for example monthly, quarterly);
  - balance computational cost against responsiveness;
- triggered retraining:
  - use performance alarms or drift indicators to trigger retraining;
  - thresholds on error metrics, residual tests, or drift detectors.

Trade-offs:
- stability:
  - keeping a model unchanged avoids frequent parameter changes;
  - may be preferable when drift is weak or intermittent;
- adaptability:
  - changing models too often can lead to instability and overfitting to short-term noise;
  - careful design of retraining criteria is needed.

Simulation-based studies can be useful here: for example, students can generate synthetic series
with known drift patterns and compare how different retraining policies affect performance under
temporal validation with various gap lengths.


### 3.5 Core message: maintenance, not train-once-and-forget

Deployment for time series models is an ongoing process:
- models must be monitored and maintained;
- adaptations should be documented and justified;
- drift and regime changes must be taken seriously.

The core message is:
- train-once-and-forget is rarely adequate in drifting environments;
- evaluation and uncertainty must be revisited over time;
- causal intelligence can help interpret detected drift and inform interventions.

This connects explicitly to the Evaluation and Uncertainty and Causal Intelligence modules:
- Evaluation and Uncertainty:
  - focuses on robust metrics, uncertainty quantification, and performance monitoring;
- Causal Intelligence:
  - addresses interventions, regime indicators, and causal graphs that can explain and respond to
    drift.


In [ ]:
# See Companion C Section 2 (and Section 6 for simulations) for code that tracks performance
# over time under drift, and Companion C Section 7 for walk-forward evaluation of ML models.

### 3.6 Case study: concept drift in socio-technical systems (AI shopping agents)

The ideas in Sections 3.1–3.5 may feel abstract. This short case study
shows how concept drift can arise in a modern socio-technical system,
where the data-generating process depends on people, interfaces,
business rules, and increasingly on other ML or AI systems.

A concrete example comes from click-sequence data on commercial
websites.

Historically, models were trained on clickstreams generated by human
visitors:

- sequences of page views and clicks,
- dwell times and scroll behaviour,
- transitions from search to product pages to cart to purchase.

Features such as 'number of pages viewed so far', 'time since previous
click', or 'position in the funnel' were reasonably stable and
predictive of human behaviour.

Now consider what happens if a substantial fraction of 'users' are AI
agents (shopping assistants, automated comparison tools, browser
extensions) that:

- automatically browse many products to compare prices or features,
- follow more systematic or scripted navigation patterns,
- issue bursts of requests at times of day that differ from typical human activity,
- may place orders through APIs or scripted interactions rather than traditional browsing.

Even if the nominal task is the same (for example predicting
conversion or recommending products), the effective data-generating
process has changed:

- the distribution of click sequences and dwell times is different (for example more regular, more
  bursty, or less 'hesitant' than human behaviour);
- some features that used to be informative (for example fine-grained dwell time on a product page
  as a proxy for interest) may become less predictive;
- new patterns (for example specific timing signatures of agent traffic, metadata about user-agents,
  or device characteristics) may become more relevant.

A model trained on historical human clickstreams can become misaligned
with current behaviour when a significant proportion of traffic is
mediated by AI agents.

From a drift-monitoring perspective:

- block-wise performance curves (for example RMSE, accuracy, AUC) may start to degrade over
  time;
- the model may systematically under- or over-predict certain behaviours (for example
  overestimating human-like hesitation, underestimating agent-driven exploration);
- retraining on recent data may appear to 'fix' metrics, but without domain knowledge you may not
  realise that a qualitatively new type of user has emerged.

This example reinforces two points from the previous subsections:

1. **Drift detection is necessary but not sufficient.**
   Monitoring performance over time can alert you that something has changed, but it does not, by
   itself, explain what changed. In the clickstream example, the change is partly driven by the
   introduction and increasing use of AI agents.

2. **Domain knowledge and system context are essential.**
   When you observe drift signals (for example rising error, changing feature importances, or
   shifting residual patterns), you need to connect them back to changes in the real system: new
   policies, interface redesigns, marketing campaigns, or new types of users (including
   AI-mediated ones).

For your own mini-projects, you do not need to build AI agents or web
analytics models. The key takeaway is that:

> Concept drift often reflects changes in the wider socio-technical
> system. Performance monitoring and retraining strategies must be
> combined with an understanding of how the system and its users
> (human or AI) are evolving over time.



## 4. Multivariate and nonlinear dependence

Multivariate time series record multiple variables at each time point. They can reveal rich
interactions across variables, but also increase the risk of spurious correlations and overfitting.
TS1 Section 2.2 introduced univariate versus multivariate series; here we focus on dependence and
tools for exploring it.


### 4.1 Multivariate series and cross-dependencies

In multivariate series, we observe vectors:

$$
X_t
= \bigl(X_{1t}, X_{2t}, \dots, X_{dt}\bigr),
$$

where $d$ is the number of variables recorded at time $t$. Examples include:
- multiple sensors on an industrial system;
- collections of economic indicators;
- environmental measurements (temperature, humidity, wind, precipitation) at the same location.

Cross-dependencies across components can:
- improve forecasting (for example exploiting leading indicators);
- help interpret anomalies (for example joint changes across variables);
- support causal reasoning (for example intervention effects across a system).

Challenges:
- dimensionality:
  - many variables increase the risk of overfitting and spurious findings;
- interpretation:
  - complex interactions can be hard to explain to domain experts;
- validation:
  - temporal and multivariate dependence complicate experimental design.


### 4.2 Spurious correlations in time series

Transformations and decompositions are important when investigating
time series, as emphasised in TS1 Sections 5 and 6 and TS2 Sections 3
and 7. Points that can be emphasised include:

- naive use of models on raw non-stationary data can give misleading results:
  - random-walk examples illustrate how apparent predictive performance on levels can simply
    reflect persistence and shared trends;
  - classical 'spurious regression' examples show high $R^2$ and significant-looking coefficients
    between unrelated trending series;
- investigating derived series:
  - differences, log-transforms, or other physically motivated transforms;
  - residuals after removing trend and seasonality;
- aiming for approximate stationarity:
  - in some cases a stationary series can be obtained from a non-stationary series, for example via
    differencing as in ARIMA models;
- decomposition:
  - splitting a series into trend, seasonality, and noise (residuals) helps to focus modelling on
    the most promising structures.

Spurious correlations often arise from:
- shared trends or common external drivers;
- misaligned time indices;
- autocorrelation structures that create apparent lag relationships without causal relevance.

Careful transformation, decomposition, and stationarity checks are
essential to reduce the risk of misleading multivariate results.


### 4.3 Tools: cross-correlation and Granger-style causality

Cross-correlation and Granger-style tools provide initial probes of
multivariate dependence and predictive relationships:

- cross-correlation:
  - measures linear correlation between $X_t$ and $Y_{t+h}$ at different lags $h$;
  - is symmetric and descriptive: it reveals linear dependence patterns, not causal direction;
  - can suggest lead–lag relationships and potential predictive influence;
  - is usually more interpretable after making series approximately stationary (for example via
    differencing or removing trend/seasonality).

- Granger-style predictive causality:
  - asks whether including past values of $X$ improves forecasts of $Y$ beyond using $Y$'s own
    past in a specified model class;
  - in that model class, if $X$ helps predict $Y$ in this sense, $X$ is said to Granger-cause $Y$;
  - the test is sensitive to model specification, lag choice, and transformations of the series.

It is crucial to remember:
- Granger-style influence is about incremental predictive content in a particular model, not
  definitive causal effect in the real world;
- confounding, common drivers, measurement changes, and structural breaks can all distort these
  relationships;
- temporal validation, careful transformations (for example to approximate stationarity), and robust
  modelling are needed to interpret such findings safely.

The existence of an algorithm does not imply that it is the right tool
to use. It is necessary to know both when an algorithm can be used and
how it should be used. This applies in particular to machine-learning
methods for time series.

Once a time-series problem has been cast into a supervised learning
task (TS2 Section 5), several flexible architectures are commonly
used, for example:
- tree-based models such as Random Forest and gradient boosting;
- recurrent or convolutional neural networks;
- additive models such as Prophet.

These methods can capture nonlinear and multivariate dependence, but
their usefulness still depends critically on:
- appropriate transformations and feature construction (for example differences, residuals,
  regime indicators);
- honest time-aware validation;
- clear comparison to simple baselines.

Companion C Section 3 provides simple templates for:
- computing and plotting cross-correlation functions (CCFs) between two series;
- running basic Granger-style predictive tests using `statsmodels`.

These are intended as exploratory tools; as emphasised here,
interpretation must be cautious and supported by transformations,
decompositions, and domain knowledge.


### 4.4 Nonlinear dependence and time-frequency ideas

Linear autocorrelation and classical spectral methods for weakly stationary series describe:
- second-order (variance-based) dependence;
- periodic structures captured by the spectral density $f(\omega)$ (TS2 Section 4.4).

However, they can miss nonlinear dependence and complex time-frequency
behaviour. High-level ideas include:
- nonlinear spectral methods:
  - defining alternative spectral-like quantities, such as quantile spectra, constructed using
    quantile-regression-style dependence rather than only second moments;
  - capturing dependence patterns beyond simple variance;
- time-frequency representations:
  - spectrogram-like plots, wavelets, and related tools;
  - highlighting how frequency content evolves over time.

These methods are typically research tools rather than standard
project requirements. They often require careful choices of tuning
parameters and are sensitive to non-stationarity, but are useful to
know about as potential directions for PhD work or advanced reading.


### 4.5 Tags, metadata, and contextual features

Additional information (tags, metadata, context variables) can be
incorporated into time-series models for forecasting, anomaly
detection, and drift monitoring. Examples include:

- calendar effects:
  - day of week, holidays, working days versus weekends;
- events:
  - campaigns, outages, maintenance periods;
- operational modes:
  - different regimes or configurations of a system;
- sensor metadata:
  - location, type, calibration status.

Given relevant transformations of a time series sample, we can apply
standard supervised-learning tools such as:
- decision trees;
- Random Forest;
- gradient boosting methods.

Companion C Section 4 gives a concrete pattern for turning a target
series and aligned exogenous/context features into a
supervised-learning dataset with lagged inputs, and for fitting
tree-based models such as `RandomForestRegressor` and histogram-based
gradient boosting. This mirrors the forecasting-with-lags formulation
from TS2 Section 5, extended to include contextual features.

In many cases it is necessary to:
- transform the time series problem into a supervised-learning problem before applying these
  methods (TS2 Section 5);
- use walk-forward or rolling-origin validation to avoid leakage and overoptimistic estimates
  (TS2 Section 6).

These points tie naturally into the need to align tags, metadata, and context features correctly in
time:
- avoid using future context information for past predictions;
- ensure that regime indicators and events are recorded accurately and synchronised with the series;
- treat static and temporal features consistently in multivariate models.

Simulation-based mini-projects can also explore these issues by
constructing multivariate synthetic data with known cross-dependencies
and testing whether simple tools recover the intended structure under
time-aware validation.


In [ ]:
# TODO: simple multivariate example with cross-correlation and basic Granger-style exploration
# TODO: optional short example slide referring to Companion C Section 4 (lag + exogenous ML) and
#       Section 7 (walk-forward evaluation) without duplicating code.

## 5. Explainability and cross-module links

Explainability for time-series models concerns:
- understanding why forecasts or anomaly flags are produced;
- relating model behaviour to domain concepts and causal reasoning.

This section links these ideas to other PCS956 modules.


### 5.1 Understanding forecasts and anomaly flags

For models based on lagged features (trees, ensembles, linear models), common tools include:
- feature importance measures:
  - impurity-based importance for tree-based models (built into many libraries);
  - permutation importance, which perturbs one feature at a time and is often more reliable for
    interpretation;
  - coefficients for linear models, possibly regularised;
- partial dependence plots or similar summaries:
  - show how forecasts change as specific lags or context variables vary.

Companion C Section 5 includes basic feature-importance tools for
tree-based models fitted on lagged and contextual features, allowing
students to identify which lags and context variables most influence
forecasts or anomaly flags.

For sequence models (for example recurrent or attention-based networks), more advanced attribution
techniques can be used:
- attention weights in models with explicit attention mechanisms;
- saliency-style methods and gradient-based attribution.

Understanding why a model raises an alarm or forecasts a certain pattern is crucial for:
- trust and decision-making;
- debugging models and data;
- communicating results to domain experts and stakeholders;
- deciding whether anomaly flags or drift alarms should trigger real-world actions.


### 5.2 Links to Causal Intelligence

Temporal order, interventions, regime shifts, and causal graphs are central in causal reasoning for
time series. Anomalies and drift naturally trigger causal questions:
- what changed and when;
- which variables were affected;
- which interventions might prevent or mitigate similar events.

Lagged relationships and regime indicators can be used as inputs to causal models:
- causal graphs that include time-indexed variables;
- intervention analysis where policies or treatments are introduced at specific times;
- structural models that distinguish causal effects from mere predictive relationships.

Lecture TS3 provides the predictive and anomaly-oriented perspective; the Causal Intelligence module
provides tools to reason about mechanisms and interventions.


### 5.3 Links to Explainable AI and Evaluation and Uncertainty

Explainable AI and Evaluation and Uncertainty share many themes with time-series work:

- trustworthy predictions:
  - forecasts should be accompanied by uncertainty intervals and clear assumptions;
  - anomaly flags should be interpretable and robust to small perturbations;
- deployment monitoring:
  - performance-over-time plots, drift detection, and alarm systems;
  - feedback loops to update models and thresholds.

Explainability techniques for time-series models (feature importance, attribution, attention) connect
directly to broader Explainable AI ideas. Drift monitoring and performance tracking connect to
Evaluation and Uncertainty, highlighting shared concerns about robustness and reliability.


In [ ]:
# TODO: example of simple feature importance or attribution for a time-series model
#       (can be aligned with Companion C Section 5 rather than re-implementing code here).

## 6. Research trends (high-level)

This section gives brief pointers to selected research directions
relevant to time series and machine learning. The aim is to indicate
areas of current activity rather than to provide a comprehensive
survey or to prescribe particular tools.


### 6.1 Causal decompositions

One example of advanced work combining decomposition, information
measures, and causality is:
- a Causal Multivariate Empirical Mode Decomposition framework for multiple time-series feature
  selection in forecasting.

Such work typically:
- decomposes a multivariate system (for example inflow, precipitation, snow depth, humidity, and
  temperature variables) into components;
- models information transfer using measures such as conditional mutual information and transfer
  entropy to identify linear and nonlinear dependencies;
- uses causal significance testing to prune to a causal network;
- uses the causal graph for target-related feature selection, combined with machine-learning-based
  forecasting algorithms.

This provides one example of how causal decompositions and
information-theoretic measures can be integrated with forecasting and
feature selection in multivariate time series.


### 6.2 Nonlinear spectral methods and time-series-to-images

Classical spectral density $f(\omega)$, the Fourier transform of
autocovariances $\gamma(h)$, reveals important information about
periodic structures but does not capture nonlinear dependence
structures.

Alternative approaches define spectral-like quantities such as the
quantile spectrum, constructed using quantile-regression-style
dependence rather than only second moments. Work in this area
demonstrates how these ideas can be used in combination with deep
learning, for example:
- quantile-frequency analysis and deep learning for signal classification;
- applications to nondestructive evaluation;
- semi-parametric estimation of the quantile spectrum with applications to earthquake
  classification using convolutional neural networks.

These works:
- generalise spectral methods to capture more information about nonlinear structures;
- map time series into image-like representations (for example quantile-frequency plots);
- use convolutional neural networks for classification tasks in domains such as nondestructive
  testing and earthquake classification.

They illustrate how time-series-to-images transformations can leverage
advances in image recognition.


### 6.3 Kolmogorov–Arnold Networks (KANs)

Kolmogorov–Arnold Networks (KANs) are inspired by the
Kolmogorov–Arnold representation theorem, which states that a
multivariate continuous function on a bounded domain can be
represented as a finite composition of simpler continuous
functions. KANs:

- place learnable univariate functions (for example implemented via B-splines) on the edges of a
  network;
- aim to offer improved interpretability and parameter efficiency compared to some standard MLPs,
  RNNs, and LSTMs;
- can be adapted to univariate and multivariate time series.

Since the initial KAN proposal, a growing body of work has applied and
extended KAN-style architectures to financial time-series forecasting,
concept drift detection, and traffic or satellite-communication
forecasting.


### 6.4 Emphasis on sceptical evaluation of new methods

Information-theoretic quantities such as entropy often appear in
causal decomposition and dependency modelling. For a discrete random
variable $X$ taking values $x$ in a set $V_X$ with probabilities $p(x)
= \operatorname{Pr}(X = x)$, the entropy is:

$$
H(X)
= - \sum_{x \in V_X} p(x)\, \log\bigl(p(x)\bigr).
$$

The choice of logarithm base matters only up to a constant factor:
using base 2 gives entropy in “bits”, while using the natural
logarithm gives entropy in “nats”. For many purposes, relative
comparisons are unaffected by this choice.

While this definition may look abstract, entropy can be interpreted as
the expected value of how surprised we are when we observe the events
$X = x$. Higher entropy corresponds to more unpredictability.

There is an abundance of blogs and videos about machine learning and
time series, but it is important not to forget the research
literature. To build on existing work and avoid unnecessary
reinvention, it can be useful to:

- search scholarly databases for combinations such as machine learning and time series;
- use curated forums or blogs to identify high-quality discussions and bibliographies.

These sources can help identify, for example:

- anomaly detection and change-point methods;
- concept drift and adaptive or online learning approaches;
- multivariate and nonlinear dependence modelling techniques.

When encountering new methods, it is generally helpful to:

- compare against simple baselines;
- use proper temporal validation with awareness of drift and leakage;
- examine behaviour on pure noise and random walks;
- assess robustness across datasets and regimes;
- consider explainability and deployment implications.


## 7. Time-series mini-project: anomalies, drift, and multivariate structure

This section gives optional guidance for students who choose to
connect their mini-projects to topics from this lecture, such as
anomalies, concept drift, multivariate dependence, or simulation-based
studies. Not all projects need to address these aspects: some may
focus on more basic datasets and questions where TS3 ideas are only
partly, or not at all, relevant.

TS1 Section 8 and TS2 Section 8 introduced the project framing and
forecasting/validation support.  TS3 adds examples of how anomalies,
drift, and multivariate thinking can be incorporated, where
appropriate for the project.

The main project question does not have to be forecasting. Possible
project types include, for example:

- forecasting tasks (for example short-term demand, climate indices, financial quantities);
- classification tasks (for example event type from temporal signals);
- anomaly or change-point detection;
- simple structural modelling or simulation-based studies of temporal methods.

Simulation-based projects are also possible, for example:
- generating synthetic time series with known drift or anomaly patterns;
- investigating how gap length between training, validation, and test affects estimates of
  performance and drift signals;
- exploring how different baselines and models behave on random-walk-like versus structured series.

The subsections below are intended as a menu of ideas. Depending on
the dataset and research question, some items may be highly relevant
and others may be unnecessary.


## 7.1 Models and baselines

For most projects it is helpful to:

- include at least one simple baseline (for example persistence or seasonal naive from TS2
  Section 2);
- choose one main model (classical or ML) appropriate for the question and data;
- document why each choice is reasonable given the series structure, multivariate context (if
  any), and domain.

Baselines can:

- reflect plausible behaviour in the absence of complex structure;
- be evaluated with time-aware validation (TS2 Section 6);
- serve as reference points for interpreting more advanced models.

For projects that use simulation, baselines and models can be evaluated both on the simulated
levels and on appropriate transforms (differences, residuals), with explicit comparison to the
known data-generating mechanism.

Companion C Section 6 provides one example of scaffolding for simulation-based studies, including:
- an AR(1) simulator with controllable level and variance shifts;
- a block-wise evaluation wrapper to study how drift affects model performance and retraining
  strategies.


## 7.2 Validation, residual analysis, and documentation

Depending on the project, it may be useful to:

- run temporal validation schemes (TS2 Section 6);
- evaluate performance on levels versus differences and residuals (TS2 Section 7);
- perform drift monitoring (Section 3 of this lecture) where drift is plausibly an issue;
- check for anomalies and level shifts (TS1 Section 3 and Section 2 of this lecture) where these
  are relevant to the question.

Documentation can cover, at an appropriate level of detail:

- data quality, anomalies, and preprocessing steps;
- transformations (for example differencing, logs, decompositions);
- model assumptions and how they were checked, where applicable;
- drift considerations and any adaptation strategies used, if drift is part of the study;
- for simulation-based work, a clear description of how the synthetic data were generated and
  how this links to the questions being investigated.

It is often informative to record not only the final choice but also
what was tried, for example:

- models or approaches that were not pursued further and why;
- series or targets that appear close to random-walk-like or noise;
- unexpected anomalies or regime changes;
- sensitivity of results to choices such as gap length between training and evaluation windows.


## 7.3 Evaluating whether models go beyond persistence

For projects where forecasting performance is central, it is typically
helpful to assess whether models provide genuine predictive value
beyond persistence. This can involve:

- comparing models against persistence and seasonal baselines under temporal validation;
- checking performance on differences or residuals where appropriate;
- examining whether apparent gains on levels are simply exploiting trend and autocorrelation.

In some cases, data may be nearly random-walk-like and hard to
predict. Demonstrating that:

- baselines are hard to beat;
- more complex models offer limited additional value;

can be a meaningful scientific finding, especially when the
limitations are clearly documented and connected to random-walk
intuition (TS1 Section 6 and TS2 Section 3).


## 7.4 Expectations: scientific attitude over sophistication

For any model containing tuning parameters, it is important to ensure
that optimisation keeps them within acceptable values. An anecdotal
example from oceanography:

- a group of students implemented an advanced fluid model for the Baltic Sea;
- after extensive tuning they obtained a close match between simulated results and actual
  observations;
- however, one tuned parameter represented viscosity, and they had effectively set it to the
  viscosity of syrup.

The Baltic Sea is not filled with syrup. Despite the good fit to data,
the model was physically nonsensical.

Moral:

- check that optimisation algorithms respect domain constraints;
- ensure that fitted parameter values remain physically and scientifically plausible.

This attitude carries over to time series models:

- baselines and advanced models should be assessed not only by error metrics but also by domain
  realism;
- drift adaptation, anomaly detection, and multivariate modelling should respect basic
  constraints of the system under study.

A strong project is honest, sceptical, and well-documented. Technical
sophistication is not a requirement; clear articulation of
assumptions, baselines, limitations, and (where relevant) drift and
cross-module connections is often more important than using advanced
models.

Simulation-based projects can be one way to demonstrate this kind of
scientific attitude:

- assumptions are explicit in the data-generating mechanism;
- limitations of methods can be explored systematically;
- results can be linked back to real-world scenarios with appropriate caveats.